In [ ]:
from glob import glob
import json
import pandas as pd

graded_data = glob("../experiments/CS-Combi/*/*_grade.json")
full_data = []

for f in graded_data:
    with open(f, "r") as fp:
        d = pd.DataFrame(json.load(fp))
    d['rank'] = [int(f.split("/")[-1].replace("_grade.json", "").replace("r", ""))] * len(d)
    d['lora'] = [f.split("/")[-2]] * len(d)
    d['grade'] = d['grade'].apply(lambda x: x[0])

    full_data.append(d)

full_data = pd.concat(full_data)
full_data = full_data.astype({'grade':int})
full_data

improvements = []
for i in list(set(full_data['id'])):
    d = full_data[full_data['id']==i]

    for r in [4,8,16,32,64,128]:
        m = d[d['rank']==r]
        lora_1_grade = list(m[m['lora']=='lora-1']['grade'])[0]
        lora_2_grade = list(m[m['lora']=='lora-2']['grade'])[0]
        lora_12_grade = list(m[m['lora']=='lora-12']['grade'])[0]
        lora_1_ga = m[m['lora']=='lora-1']['generated_answer']._values[0]
        lora_2_ga = m[m['lora']=='lora-2']['generated_answer']._values[0]
        lora_12_ga = m[m['lora']=='lora-12']['generated_answer']._values[0]
        expl_1 = m[m['lora']=='lora-1']['explanation']._values[0]
        expl_2 = m[m['lora']=='lora-2']['explanation']._values[0]
        expl_12 = m[m['lora']=='lora-12']['explanation']._values[0]
        improvements.append({"id":i, "type":list(m['type'])[0], "rank":r, "query":list(m['query'])[0], "reference_answer":list(m['reference_answer'])[0], "expl-1":expl_1,"expl-2":expl_2, "expl-12":expl_12, "generated_answer_lora_1":lora_1_ga,
                             "generated_answer_lora_2":lora_2_ga,"generated_answer_lora_12":lora_12_ga,'max_grade':max(lora_1_grade, lora_2_grade), 'lora_1_grade':lora_1_grade, 'lora_2_grade':lora_2_grade, "lora_12_grade":lora_12_grade,  "improvement_over_max": lora_12_grade-max(lora_1_grade, lora_2_grade),
                             "improvement_over_min":lora_12_grade-min(lora_1_grade, lora_2_grade)})


improvements = pd.DataFrame(improvements)

improvements_ids = improvements[improvements['improvement_over_max']>0]
no_improvements_ids = improvements[improvements['improvement_over_max']<=0]

improvements_ids = improvements_ids[improvements_ids['type']=='double']
for idx, d in improvements_ids.iterrows():
    print("Question", d['query'])
    print("Reference answer", d['reference_answer'])
    print("LoRA1: " , d['generated_answer_lora_1'])
    print("LoRA2: " , d['generated_answer_lora_2'])
    print("LoRA12: " , d['generated_answer_lora_12'])
    break
# improvements_ids

print(len(improvements_ids), len(improvements))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


plt.figure(figsize=(6.5,2.3))
ax = sns.lineplot(improvements, x='rank', y='improvement_over_min', markers=True, marker="o", label="over min")
ax.set_xlabel ('LoRA Rank')
ax.set_ylabel ('Grade improvement')
ax.yaxis.set_label_coords(-0.09, 0.35)
ax = sns.lineplot(improvements, x='rank', y='improvement_over_max', markers=True, marker="o", label='over max')
ax.grid()
plt.xticks([4, 8, 16, 32, 64, 128])

plt.savefig("../figs/replica_lora_merge_both.pdf",bbox_inches='tight')